In [1]:
from build123d import *

In [2]:
import filepath_p2

In [3]:
from typing import Dict
from panda3d.core  import (
    NodePath,
    Vec3
)
from panda3d.bullet import (
    BulletRigidBodyNode,BulletConvexHullShape
)
from panda3d.bullet import BulletDebugNode,BulletWorld
from panda3d.core import *
from build123d import *
from panda3d.core import NodePath
from util.geometry import *
import torch
import importlib
from cad_tools import cadmesh
from cad_tools import matrix
from cad_tools import constraint
importlib.reload(cadmesh)
importlib.reload(matrix)
importlib.reload(constraint)
from cad_tools.cadmesh import *
from cad_tools.matrix import *
from cad_tools.constraint import *
import debug
importlib.reload(debug)
from debug import *

In [4]:
# help(BulletHingeConstraint)

In [5]:
from panda3d.bullet import BulletHingeConstraint, BulletGenericConstraint
from panda3d.core import Mat4



In [6]:
class GameObjectFromCAD:
    def __init__(self, build123d_obj):
        self.cad_obj = build123d_obj # as a single obj 
        

    def get_geom(self):
        pass

In [7]:
class BearingDisk(Compound):
    def __init__(
        self,
        diameter: float,
        thickness: float,
        bearing_diameter: float,
        bearing_depth: float,
        shaft_diameter: float,
        parent=None
    ):
        with BuildPart() as disk_part:
            # 主体圆盘
            with BuildSketch():
                Circle(diameter / 2)
            extrude(amount=thickness)

            # 轴承座（浅孔）
            with BuildSketch(Plane.XY):
                Circle(bearing_diameter / 2)
            extrude(amount=-bearing_depth, mode=Mode.SUBTRACT)

            # 中轴孔（贯穿）
            with BuildSketch(Plane.XY):
                Circle(shaft_diameter / 2)
            extrude(amount=thickness, mode=Mode.SUBTRACT)

            # ⚡ 父 joint：固定
            RigidJoint(
                label="shaft_joint",
                joint_location=Location((0, 0, thickness / 2),(0,0,90))
            )

        super().__init__(
            disk_part.part.wrapped,
            joints=disk_part.part.joints,
            parent=parent
        )


In [8]:
import torch
from util.bullet_geometry import *
from util.geometry import *
from art.basic import *
def cube_frame_convex_hull(edge_length,edge_width):
    trape_edge = torch.Tensor([
        [-edge_length / 2, edge_length / 2, -edge_length / 2],
        [edge_length / 2, edge_length / 2, -edge_length / 2],
        [edge_length / 2 - edge_width, edge_length / 2 - edge_width, -edge_length / 2],
        [-edge_length / 2 + edge_width, edge_length / 2 - edge_width, -edge_length / 2]
    ])
    recta_edge = torch.Tensor([
        [-edge_length / 2 + edge_width, edge_length / 2 - edge_width, -edge_length / 2],
        [edge_length / 2 - edge_width, edge_length / 2 - edge_width, -edge_length / 2],
        [edge_length / 2 - edge_width, edge_length / 2 - edge_width, -edge_length / 2 + edge_width],
        [-edge_length / 2 + edge_width, edge_length / 2 - edge_width, -edge_length / 2 + edge_width],
    ])
    # first rotation
    transforms_1edge_to_1surf = [
            # xy
            torch.Tensor([
                [0, 1, 0],
                [-1, 0, 0],
                [0, 0, 1]
            ]),
            torch.Tensor([
                [-1, 0, 0],
                [0, -1, 0],
                [0, 0, 1]
            ]),
            torch.Tensor([
                [0, -1, 0],
                [1, 0, 0],
                [0, 0, 1]
            ]),
            torch.Tensor([
                [1, 0, 0],
                [0, 1, 0],
                [0, 0, 1]
            ])
        ]
    transforms_1surf_to_cube = [
            torch.Tensor([
                [1, 0, 0],
                [0, 1, 0],
                [0, 0, 1]
            ]),
            # xz
            torch.Tensor([
                [0, 0, 1],
                [0, 1, 0],
                [-1, 0, 0]
            ]),
            torch.Tensor([
                [-1, 0, 0],
                [0, 1, 0],
                [0, 0, -1]
            ]),
            torch.Tensor([
                [0, 0, -1],
                [0, 1, 0],
                [1, 0, 0]
            ]),
            # yz
            torch.Tensor([
                [1, 0, 0],
                [0, 0, -1],
                [0, 1, 0]
            ]),
            torch.Tensor([
                [1, 0, 0],
                [0, 0, 1],
                [0, -1, 0]
            ])
        ]
    convex_shapes = []
    trape_surf = batch_transform(
        [trape_edge], transforms_1edge_to_1surf)
    trape_cube = batch_transform(
        trape_surf, transforms_1surf_to_cube)
    recta_surf = batch_transform(
        [recta_edge], transforms_1edge_to_1surf)
    recta_cube = batch_transform(
        recta_surf, transforms_1surf_to_cube)
    # FIXME: use minkowski sum
    for i in range(24):
        # print(i)
        convex_shape = BulletConvexHullShape()
        for point in trape_cube[i]:
            convex_shape.addPoint(Vec3(*point))
        for point in recta_cube[i]:
            convex_shape.addPoint(Vec3(*point))
        convex_shapes.append(convex_shape)
    return convex_shapes
    


In [9]:
class Gimbal(Compound):
    def __init__(
        self, 
        disc_diameter_gap: float,
        disc_thickness_gap: float,
        frame_thickness_diameter: float, 
        corner_radius: float,
        connector_length: float,
        connector_radius: float,
        # disk
    ):
        """
        创建一个用于支撑转盘的 Gimbal 框架
        disc_diameter_gap: 转盘直径 + 留出间隙
        disc_thickness_gap: 转盘厚度 + 留出间隙
        frame_thickness_diameter: 矩形管管径
        corner_radius: 方孔圆角半径
        connector_length: 左右连接柱长度
        connector_radius: 左右连接柱半径
        """
        a = disc_diameter_gap + frame_thickness_diameter/2
        b = disc_thickness_gap + frame_thickness_diameter/2
        r = corner_radius
        tube_radius = connector_radius
        tube_length = connector_length
        # self.disk = disk

        with BuildPart() as gimbal_part:
            # -----------------------------
            # 1️⃣ 构建圆角矩形路径
            with BuildLine() as frame:
                start = (-a/2 + r, -b/2)
                Line(start, (a/2 - r, -b/2))
                JernArc(start=(a/2 - r, -b/2), tangent=(1, 0), radius=r, arc_size=90)
                Line((a/2, -b/2 + r), (a/2, b/2 - r))
                JernArc(start=(a/2, b/2 - r), tangent=(0, 1), radius=r, arc_size=90)
                Line((a/2 - r, b/2), (-a/2 + r, b/2))
                JernArc(start=(-a/2 + r, b/2), tangent=(-1, 0), radius=r, arc_size=90)
                Line((-a/2, b/2 - r), (-a/2, -b/2 + r))
                JernArc(start=(-a/2, -b/2 + r), tangent=(0, -1), radius=r, arc_size=90)
            rectangle_wire = frame.wire()

            # -----------------------------
            # 2️⃣ sweep 圆角矩形管
            # TODO: start and tangent
            start = rectangle_wire @ 0
            tangent = rectangle_wire % 0 
            with BuildSketch(Plane(origin=start, z_dir=tangent)) as sk:
                Circle(frame_thickness_diameter/2)
            sweep(sk.sketch, rectangle_wire, is_frenet=True)

            # -----------------------------
            # 3️⃣ 左右中点圆柱（extrude）沿 X 方向
            left_mid = (-a/2, 0)
            right_mid = (a/2, 0)
        
            for mid, direction in [(left_mid, -1), (right_mid, 1)]:
                plane = Plane(origin=(mid[0], mid[1], 0), z_dir=(direction, 0, 0))
                with BuildSketch(plane) as skc:
                    Circle(connector_radius)
                extrude(skc.sketch, connector_length)

            # debug
            thick_radius = connector_radius * 1.8  # 更粗

            plane = Plane(origin=(a/2, 0, 0), z_dir=(1, 0, 0))
            with BuildSketch(plane) as sk:
                Circle(thick_radius)
            extrude(sk.sketch, connector_length)
                    
            # debug 
            box_size = 1

            with BuildPart() as extra:
                Box(
                    box_size,
                    box_size,
                    box_size,
                    align=(Align.MIN, Align.CENTER, Align.CENTER),
                )
            
                # 移到右侧 + 偏 Y
                extra.part.move(Location((0, .2, .2)))
            # debug finish
           #constraints 
            RevoluteJoint(
                label="bearing_axis",
                axis=Axis((0, 0, 0), (0, 1, 0)),
                angular_range=(0, 360)
            )                         
        # gimbal_part.part.joints["bearing_axis"].connect_to(disk.joints["shaft_joint"])
        super().__init__(
            gimbal_part.part.wrapped,
            joints=gimbal_part.part.joints,
            # children=[disk]
        )
        


In [10]:
from panda3d.core import Mat4, LVector3f

# 假设平移向量 (x, y, z)
translation = LVector3f(5, -2, 3)

# 创建平移矩阵
m = Mat4.translateMat(translation)
m

LMatrix4f(1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 5, -2, 3, 1)

In [11]:
import numpy as np

In [12]:
from cad_tools.cadmesh import *

In [13]:
from build123d import *
import numpy as np

class Frame(Compound):
    def __init__(self, cube_size, beam,label=None):
        box = Box(cube_size, cube_size, cube_size)
        center = Vector(0, 0, 0)
        joints = []
        with BuildPart() as bp:
            # ========= 1️⃣ 生成框架 =========
            for edge in box.edges():
                mid = edge.position_at(0.5)
                direction = edge.tangent_at(0)
                outward = (mid - center).normalized()
                plane = Plane(
                    origin=mid - outward * (beam / np.sqrt(2)),
                    z_dir=direction
                )
                with BuildSketch(plane):
                    Rectangle(beam, beam)
                sweep(path=edge)

        
        # ========= 2️⃣ 添加 joints =========
        # 六个面
        # face_dirs = [
        #     Vector(1,0,0), Vector(-1,0,0),
        #     Vector(0,1,0), Vector(0,-1,0),
        #     Vector(0,0,1), Vector(0,0,-1),
        # ]
        
        # for i, normal in enumerate(face_dirs):
        #     face_center = normal * (cube_size / 2)
            
        #     joint = RevoluteJoint(
        #         label=f"face_{i}",
        #         parent=bp.part,
        #         location=Location(
        #             Plane(
        #                 origin=face_center,
        #                 z_dir=normal  # 👉 旋转轴方向
        #             )
        #         )
        #     )
            
        #     joints.append(joint)
            x_faces = [
                (Vector(cube_size/2, 0, 0), (0,0,1)),
                (Vector(-cube_size/2, 0, 0), (0,0,-1))
            ]
            for i, (face_center, axis) in enumerate(x_faces):
                RevoluteJoint(
                    label=f"x_face_{i}",
                    axis=Axis(face_center, axis),
                    angular_range=(-90, 90)
                )
        
        # ========= 3️⃣ 初始化 Compound =========
        super().__init__(
            bp.part.wrapped,
            joints=bp.part.joints,
            label=label
        )



In [14]:
sizes = {
    'cube_size' : 100,
    'beam_size' : 5,
    
    'disk_diameter' : 50,
    'disk_thickness': 5,
    'frame_thickness_diameter': 5,
    'gap': 4,
    'corner_radius':2,
    'connector_length':20,
    'connector_radius':2,

    'bearing_diameter': 22.2,
    'bearing_depth': 7,
    'shaft_diameter': 8.2
}
sizes = {
    k:v/10
    for k,v in sizes.items()
}

In [15]:
import math
import time
import math
def normalize_angle(a):
    return (a + math.pi) % (2 * math.pi) - math.pi
last_print_time = 0.0
print_interval = 0.5  # 秒

# target_angle = math.radians(30)  
target_angle=30
def update_hinge(hinge, target_angle=target_angle):
    global last_print_time
    # current = hinge.getHingeAngle()
    
    current = normalize_angle(hinge.getHingeAngle())
    error = target_angle - current
    now = time.time()
    


    # 简单比例控制（P controller）
    kp = 3.0
    speed = kp * error

    # 限制最大速度（防止抖动）
    max_speed = 200
    speed = max(-max_speed, min(max_speed, speed))
    if now - last_print_time >= print_interval:
        print("target", target_angle)
        print("current angle:", math.degrees(current))
        last_print_time = now
        print("speed",speed, kp * error)

    # hinge.enableAngularMotor(True, speed, 500)
    # motor.setMaxMotorForce(.5)

    # motor.setMotorSpeed(speed)
def print_hinge_angle(hinge):
    current = normalize_angle(hinge.getHingeAngle())
    print(current)

In [16]:
from panda3d_game.game_object import GameObject,PhysicsGameObject

class FrameObj(PhysicsGameObject):

    def __init__(self,name,mass=1):
        PhysicsGameObject.__init__(self)
        self.name = f"Frame.{name}"
        self.mass = mass
        self.f_cad = Frame(
            sizes['cube_size'],
            sizes['beam_size'],label="frame")
        self.rigid_node = BulletRigidBodyNode(self.name)
        self.rigid_node.setMass(self.mass)
        # TODO: convex hull: 
        convex_shapes = cube_frame_convex_hull(
            edge_length=sizes["cube_size"],
            edge_width=sizes["beam_size"]
        )
        for shape in convex_shapes:
            self.rigid_node.addShape(shape)
        self.rigid_np = NodePath(self.rigid_node)
        self.rigid_node.setLinearSleepThreshold(0) # 
        self.rigid_node.setAngularSleepThreshold(0) #
        self.rigid_node.setFriction(0)


        # self.rigid_node.setAngularDamping(10.9)
        # self.rigid_node.setLinearDamping(10.9)
        

        
    @property
    def joints(self):
        return self.f_cad.joints

    def render(self,*args, **kwargs):
        mesh_node = build123d_to_panda_node(
            name=self.name,
            part=self.f_cad.solid(), format_=format_default)
        mesh_np = NodePath(mesh_node)
        mesh_np.reparentTo(self.mainPath)
        try:
            min_pt, max_pt = mesh_np.getTightBounds()
            print(min_pt,max_pt)
        except Exception as e:
            print(e)

    def compounds(self):
        return self.f_cad

class CMGGimbal(PhysicsGameObject):

    def toBulletWorld(self, bullet_world: BulletWorld):
        bullet_world.attachRigidBody(self.rigid_node)
        bullet_world.attachRigidBody(self.disk_rigid_node)
        for constraint in self.constraints:
            bullet_world.attachConstraint(constraint)
        for c in self.children:
            try:
                c.toBulletWorld(bullet_world)
            except Exception as e:
                self.log(e)

    def setMat(self, *args, **kwargs):
        self.disk_rigid_np.setMat(*args,**kwargs)
        self.rigid_np.setMat(*args,**kwargs)

    def reparentTo(self, *args, **kwargs):
        self.disk_rigid_np.reparentTo(*args,**kwargs)
        self.rigid_np.reparentTo(*args,**kwargs)
        

                
    def __init__(self,name):
        PhysicsGameObject.__init__(self)
        self.name = f"cmg_gimbal.{name}"
        self.gimbal_cad = Gimbal(
                disc_diameter_gap=sizes['disk_diameter']+sizes['gap'], 
                disc_thickness_gap=sizes['disk_thickness']+sizes['gap'], 
                frame_thickness_diameter=sizes['frame_thickness_diameter'], 
                corner_radius=sizes['corner_radius'], 
                connector_length=sizes['connector_length'], 
                connector_radius=sizes['connector_radius'],
            )
        self.disk_cad = BearingDisk(
                diameter=sizes['disk_diameter'],
                thickness=sizes['disk_thickness'],
                bearing_diameter=sizes['bearing_diameter'],
                bearing_depth=sizes['bearing_depth'],
                shaft_diameter=sizes['shaft_diameter'],
            )
        self.gimbal_cad \
                .joints["bearing_axis"] \
                .connect_to(self.disk_cad.joints["shaft_joint"])
        self.compound_cad = Compound(
                label=f"gc_{name}", children=[self.gimbal_cad,self.disk_cad], 
            ) 
        self.compound_cad.joints["gimbal_rotation"] = RigidJoint(
                label="gimbal_rotation",
                joint_location=Location((0, 0, 0),(0,90,90)),
                to_part=self.compound_cad
            )
        self.rigid_node = BulletRigidBodyNode(self.name) # node for gimbal
        self.rigid_node.setMass(.1)
        self.rigid_np = NodePath(self.rigid_node)
        
        self.disk_rigid_node = BulletRigidBodyNode(f"{self.name}.disk")
        self.disk_rigid_node.setMass(1)
        self.disk_rigid_np = NodePath(self.disk_rigid_node)
        # self.disk_rigid_np.reparentTo(self.rigid_np)

        # get transform of disk relative to gimbal compound in cad
        disk_transform = trsf_to_mat4(
            self.disk_cad.location.wrapped.Transformation()
        )
        # self.disk_rigid_np.setMat(self.rigid_np, disk_transform)

        # convec_hull_disk = create_convex_hull_shape_tr(
        #     geoms = [build123d_to_panda_geom(self.disk_cad.solid(), format_=format_default)], # TODO get geom TODO tol=
        #     transforms = [disk_transform]
        # )
        convec_hull_disk = create_convex_hull_shape(
            geoms = [build123d_to_panda_geom(self.disk_cad.solid(), format_=format_default)], # TODO get geom TODO tol=
        )
        # TODO: check pos 
        self.disk_rigid_node.addShape(convec_hull_disk)
        
        # TODO: set constraint
        revolute_j = self.gimbal_cad.joints["bearing_axis"]
        self.bearing_constraint = joint_to_bullet_constraint(
            revolute_j, parent_node=self.rigid_np, child_node=self.disk_rigid_np,debug=True
        )
        self.constraints = [
            self.bearing_constraint
        ]
        for node in [self.disk_rigid_node, self.rigid_node]:
            node.setLinearSleepThreshold(0)
            node.setAngularSleepThreshold(0)
            node.setFriction(0)


            
        
        

    def render(self,*args,**kwargs):
        mesh_disk = build123d_to_panda_node(
            name=f"{self.name}.disk.mesh", # FIXME names
            part=self.disk_cad.solid(), format_=format_default)
        mesh_gimbal = build123d_to_panda_node(
            name=f"{self.name}.gimbal",
            part=self.gimbal_cad.solid(), format_=format_default)
        mesh_disk_np = NodePath(mesh_disk)
        mesh_gimbal_np = NodePath(mesh_gimbal)
        # mesh_disk_np.reparentTo(self.disk_rigid_np)
        mesh_gimbal_np.reparentTo(self.rigid_np)
        
    @property
    def joints(self):
        return self.compound_cad.joints

    def compounds(self):
        return self.compound_cad
        
        

class ScissorPairCMG(PhysicsGameObject):
    def toBulletWorld(self, world):
        for child in self.children:
            child.toBulletWorld(world)
        # self.mainPath
        for constraint in self.constraints:
            world.attachConstraint(constraint)
    def __init__(self,name):
        PhysicsGameObject.__init__(self)
        self.name = f"sc_cmg.{name}"
        self.gimbals = [
            CMGGimbal(f"{self.name}.gimbal.{i}")
            for i in range(2)
        ]
        self.frame = FrameObj(f"{self.name}.frame")
        for i in range(2):
            self.frame.joints[f"x_face_{i}"].connect_to(
                self.gimbals[i].joints["gimbal_rotation"], 
                angle=0)
        # set transforms 
        # self.rigid_np = self.frame.rigid_np
        for i in range(2):
            transform = trsf_to_mat4(
                self.gimbals[i].compound_cad.location.wrapped.Transformation()
            )
            self.gimbals[i].setMat(transform)      
        
        self.children = [
            self.frame, 
            *self.gimbals 
        ]
        self.gimbal_constraints = {}
        for i in range(2):
            revolute_j = self.frame.joints[f"x_face_{i}"]
            print("gimbal constrain",i)
            self.gimbal_constraints[f"x_{i}"] = joint_to_bullet_constraint(
                revolute_j, 
                parent_node=self.frame.rigid_np, 
                child_node=self.gimbals[i].rigid_np,
                debug=True
            )
            
        self.constraints = [
            *self.gimbal_constraints.values()
            # self.gimbal_constraints["x_0"]
        ]
    

    def reparentTo(self, other): # rendering
        for child in self.children:
            child.reparentTo(other)

    def show_constraints(self,render):
        for i in range(2):
        # for i in [0]:
            bodyA_np = self.frame.rigid_np
            bodyB_np = self.gimbals[i].rigid_np
            hinge = self.gimbal_constraints[f"x_{i}"]
            visualize_constraint(
                world_np=render,
                    bodyA_np=bodyA_np,
                    bodyB_np=bodyB_np,
                    constraint=hinge,
                    scale=2.0
                )

    def render(self,*args,**kwargs):
        for child in self.children:
            child.render()
    
    def compounds(self):
        return Compound(
            label="cmg",
            children=[
                self.frame.compounds(),
                self.gimbals[0].compounds(),self.gimbals[1].compounds()])
        

In [17]:
from panda3d.core import NodePath, LVector3f
from panda3d.core import AmbientLight, DirectionalLight
# from panda3d.stl import load_stl
from direct.showbase.ShowBase import ShowBase
from panda3d_game.app import ContextShowBase, PhysicsShowBase
# prc = 
# prc = """

# load-display pandagl
# from 

# """
# loadPrcFileData("",prc)

class MyApp(PhysicsShowBase):
    def __init__(self, debug=True):
        
        PhysicsShowBase.__init__(self)
        # 加载 STL 文件
        # stl_node = load_stl("example.stl")  # 返回 GeomNode
        # self.model = NodePath(stl_node)
        # self.model = mesh_np
        # self.model.reparent_to(self.render)
        s = ScissorPairCMG("s")
        s.reparentTo(self.render)
        s.render()

        # 缩放 / 平移 / 旋转
        # self.model.set_scale(0.01)  # 根据 STL 单位调整
        # self.model.set_pos(0, 10, 0)
        # self.model.set_hpr(0, 0, 0)

        # 添加光源
        ambient = AmbientLight("ambient")
        ambient.set_color((0.5, 0.5, 0.5, 1))
        ambient_np = self.render.attach_new_node(ambient)
        self.render.set_light(ambient_np)

        directional = DirectionalLight("dir")
        directional.set_color((1, 1, 1, 1))
        directional_np = self.render.attach_new_node(directional)
        directional_np.set_hpr(-30, -60, 0)
        self.render.set_light(directional_np)

        # 设置背景色
        self.set_background_color(0.1, 0.1, 0.1)
        s.toBulletWorld(self.bullet_world)
        # print("disk pos")
        # print("rigid",s.gimbals[0].disk_rigid_np.getPos(self.render))
        # print("shape",
        # self.accept('p', self.pause_switch)
        self.paused=False
        print(self.paused)
        if debug:
            print("debug")
            debug_node = BulletDebugNode('Debug')
            debug_node.showWireframe(True)   # 显示线框
            debug_node.showConstraints(True) # 显示约束
            debug_node.showBoundingBoxes(True)
            debug_node.showNormals(False)
            
            debug_np = NodePath(debug_node)
            debug_np.reparentTo(render)
            debug_np.show()
            self.bullet_world.setDebugNode(debug_node)
            print("bodies")
            bodies = self.bullet_world.getRigidBodies()

            for body in bodies:
                print(body)
            s.show_constraints(self.render)
            # dt = 1/60  # 时间步长（秒）
            # self.bullet_world.doPhysics(dt)
            # print("bodies")
            # bodies = self.bullet_world.getRigidBodies()
        for i in range(2):
            print(s.gimbals[i].rigid_np.getMat())

        def hinge_task(task):
            for i in range(2):
                # s.gimbals[i].bearing_constraint
                # print(i)
                # angle = 30 if i==0 else -30
                angle=30
                # print("angle",angle)
                update_hinge(s.gimbal_constraints[f"x_{i}"],target_angle=angle)
            return task.cont
        taskMgr.add(hinge_task, "hinge_debug")
        for i in range(2):
            
            print_hinge_angle(s.gimbal_constraints[f"x_{i}"])
            # for body in bodies:
            #     print(body)
        s.frame.rigid_node.setLinearVelocity((0, 0, 0))
        s.frame.rigid_node.setAngularVelocity((0, 0, 0))


In [18]:
from qpanda3d import  QPanda3DWidget,  Synchronizer, QControlMultiView
from demos.physics_room import PhyscRoomConsole
class ShaderControl(MyApp, QControlMultiView):
    def __init__(self):
        QControlMultiView.__init__(self)
        MyApp.__init__(self)

    def pause_switch(self):
        pass

from demos.physics_room import PhyscRoomConsole
from ui.qtui import *
from demos.physics_room import PhyscRoomConsole




class ShaderGame(MultiViewQtGUI):
    def get_game(self):
        game = ShaderControl()
        # self.cameras["test"] = game.new_cam_np
        return game

    def get_console(self):
        return PhyscRoomConsole(showbase=self.panda3d)

if __name__ == '__main__':
    # torch.set_printoptions(precision=16, sci_mode=False)
    import sys
    # app = QApplication(sys.argv)
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    
    # try:
    window = ShaderGame()
    window.show()
    # except Exception as e:
        # print(e)
    # finally:
    sys.exit(app.exec_())

:display: loading display module: libpandagl.so
Known pipe types:
  glxGraphicsPipe
(all display modules loaded.)
:ShowBase: Default graphics pipe is glxGraphicsPipe (OpenGL).
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
:display: Created output of type glxGraphicsBuffer
:ShowBase: Successfully opened window of type glxGraphicsBuffer (OpenGL)
:audio: NullAudioManager
:audio: NullAudioManager
/mnt/D/Games/GameNotesPanda3D/p1/py_src/util/maths/differential.py:103: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  face_normal = torch.cross(e1, e2)


create hinge for: cmg_gimbal.sc_cmg.s.gimbal.0 cmg_gimbal.sc_cmg.s.gimbal.0.disk
frame a; frame b; frm build123d
0 0 1 0
1 0 0 0
0 1 0 0
0 0 0 1

0 0 1 0
1 0 0 0
0 1 0 0
0 0 0 1


create hinge for: cmg_gimbal.sc_cmg.s.gimbal.1 cmg_gimbal.sc_cmg.s.gimbal.1.disk
frame a; frame b; frm build123d
0 0 1 0
1 0 0 0
0 1 0 0
0 0 0 1

0 0 1 0
1 0 0 0
0 1 0 0
0 0 0 1


gimbal constrain 0
create hinge for: Frame.sc_cmg.s.frame cmg_gimbal.sc_cmg.s.gimbal.0
frame a; frame b; frm build123d
1 0 0 0
0 1 0 0
0 0 1 0
5 0 0 1

0 1 0 0
0 0 1 0
1 0 0 0
0 0 0 1


maxmin 90 -90
gimbal constrain 1
create hinge for: Frame.sc_cmg.s.frame cmg_gimbal.sc_cmg.s.gimbal.1
frame a; frame b; frm build123d
-1 0 0 0
0 1 0 0
0 0 -1 0
-5 0 0 1

0 1 0 0
0 0 1 0
1 0 0 0
0 0 0 1


maxmin 90 -90
LPoint3f(-5, -5, -5) LPoint3f(5, 5, 5)
False
debug
bodies
BulletRigidBodyNode Frame.sc_cmg.s.frame (24 shapes) active mass=1

BulletRigidBodyNode cmg_gimbal.sc_cmg.s.gimbal.0 (0 shapes) active mass=0.1 T:m(pos 5 0 0 hpr -90 0 -90)

Bulle

:display: Created output of type GLGraphicsBuffer
:pnmtext: Loaded font Perspective Sans Regular


target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0
target 30
current angle: 0.0
speed 90.0 90.0


SystemExit: 0

/mnt/D/packages/miniconda3/envs/game-qt6-py312/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import numpy as np

A = np.array([[0,0,1],
              [1,0,0],
              [0,1,0]])

A_inv = np.linalg.inv(A)
A_in

In [ ]:
e

In [ ]:
[1,*[1,2]]

In [ ]:
help(create_convex_hull_shape_tr)

In [ ]:
s = ScissorPairCMG("s")

In [ ]:
s.compounds()

In [ ]:
s.gimbals[0].compound_cad.joints["gimbal_rotation"].connected_to

In [ ]:
s.gimbals[0].compound_cad.joints["gimbal_rotation"].location

In [ ]:
s.frame.joints["x_face_0"].location

In [ ]:
s.frame.joints["x_face_1"].location

In [ ]:
s.frame.joints["x_face_1"].connected_to.location

In [ ]:
trsf_to_mat4(
                # self.gimbals[i].compound_cad.location.wrapped.Transformation()
    s.frame.joints["x_face_0"].connected_to.location.wrapped.Transformation()
            )

In [ ]:
trsf_to_mat4(
                # self.gimbals[i].compound_cad.location.wrapped.Transformation()
    s.frame.joints["x_face_0"].location.wrapped.Transformation()
            )

In [ ]:
help(RigidJoint)

In [ ]:
def revolute_joint_to_dict(j):
    return {
        "label":j.label,
        "angle_reference":j.angle_reference,
        "location":j.location,
        "angular_range":j.angular_range,
        "connected_to":j.connected_to
    # j.symbol
    }

In [ ]:
def rigid_joint_to_dict(j):
    return {
        "label":j.label,
        "location":j.location,
        "connected_to":j.connected_to
    }

In [ ]:
j = s.gimbals[0].disk_cad.joints["shaft_joint"]
print(j)
print(j.connected_to)
print("---")
j = s.gimbals[0].gimbal_cad.joints["bearing_axis"]
print(j)
print(j.connected_to)
print("---")
j2 = j.connected_to
print(j.relative_to(j2))
print(j2.relative_to(j))
print("---")
print(revolute_joint_to_dict(j))
print(rigid_joint_to_dict(j2))
# TODO: print j and j2 informations

In [ ]:
j = s.gimbals[0].disk_cad.joints

In [ ]:
l = s.gimbals[1].compound_cad.location.wrapped.Transformation()

In [ ]:
m = trsf_to_matrix(l)

In [ ]:
arr = np.array(m)

In [ ]:
np.set_printoptions(precision=3, suppress=True)
arr

In [ ]:
angle_deg = 15
c = np.cos(np.deg2rad(angle_deg))
s = np.sin(np.deg2rad(angle_deg))
c,s

In [ ]:
from build123d import Location

In [ ]:
lw = l.wrapped

In [ ]:

matrix = [
    [trsf.Value(i, j) for j in range(1, 5)] 
    for i in range(1, 4)
]
# Add the implicit bottom row for a full 4x4 homogeneous matrix
matrix.append([0, 0, 0, 1])

In [ ]:
matrix

In [ ]:
lw

In [ ]:
trsf = l.wrapped.Transformation()

In [ ]:
create_convex_hull_shape_tr

In [ ]:
cube_size = 100
beam_size = 5
g1 = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2,
)
d1 = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2,
)
g2 = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2,
)
d2 = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2,
    # parent=g2
)
g1.joints["bearing_axis"].connect_to(d1.joints["shaft_joint"], angle=50)
g2.joints["bearing_axis"].connect_to(d2.joints["shaft_joint"], angle=50)
gc1 = Compound(
    label="gc1", children=[g1,d1], 
)
gc2 = Compound(
    label="gc2", children=[g2,d2], 
)
gc1.joints["gimbal_rotation"] = RigidJoint(
    label="gimbal_rotation",
    joint_location=Location((0, 0, 0),(0,90,90)),
    to_part=gc1
)
gc2.joints["gimbal_rotation"] = RigidJoint(
    label="gimbal_rotation",
    joint_location=Location((0, 0, 0),(0,90,90)),
    to_part=gc2
)
f = Frame(cube_size,beam_size,label="frame")
f.joints["x_face_0"].connect_to(gc1.joints["gimbal_rotation"], angle=90)
f.joints["x_face_1"].connect_to(gc2.joints["gimbal_rotation"], angle=0)

In [ ]:
cmg = Compound(label="cmg",children=[f,gc1,gc2])
cmg